In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier
from sklearn.experimental import enable_halving_search_cv  
from sklearn.model_selection import HalvingRandomSearchCV, StratifiedKFold
from scipy.stats import randint, uniform, loguniform

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/heart-attack-risk-and-prediction-dataset-in-india/heart_attack_prediction_india.csv


# Load Dataset

In [3]:
df = pd.read_csv('/kaggle/input/heart-attack-risk-and-prediction-dataset-in-india/heart_attack_prediction_india.csv')

In [4]:
df.head()

,Patient_ID,State_Name,Age,Gender,Diabetes,Hypertension,Obesity,Smoking,Alcohol_Consumption,Physical_Activity,...,Diastolic_BP,Air_Pollution_Exposure,Family_History,Stress_Level,Healthcare_Access,Heart_Attack_History,Emergency_Response_Time,Annual_Income,Health_Insurance,Heart_Attack_Risk
0,1,Rajasthan,42,Female,0,0,1,1,0,0,...,119,1,0,4,0,0,157,611025,0,0
1,2,Himachal Pradesh,26,Male,0,0,0,0,1,1,...,115,0,0,7,0,0,331,174527,0,0
2,3,Assam,78,Male,0,0,1,0,0,1,...,117,0,1,10,1,0,186,1760112,1,0
3,4,Odisha,58,Male,1,0,1,0,0,1,...,65,0,0,1,1,1,324,1398213,0,0
4,5,Karnataka,22,Male,0,0,0,0,0,1,...,109,0,0,9,0,0,209,97987,0,1


In [5]:
df.shape

(10000, 26)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Patient_ID               10000 non-null  int64 
 1   State_Name               10000 non-null  object
 2   Age                      10000 non-null  int64 
 3   Gender                   10000 non-null  object
 4   Diabetes                 10000 non-null  int64 
 5   Hypertension             10000 non-null  int64 
 6   Obesity                  10000 non-null  int64 
 7   Smoking                  10000 non-null  int64 
 8   Alcohol_Consumption      10000 non-null  int64 
 9   Physical_Activity        10000 non-null  int64 
 10  Diet_Score               10000 non-null  int64 
 11  Cholesterol_Level        10000 non-null  int64 
 12  Triglyceride_Level       10000 non-null  int64 
 13  LDL_Level                10000 non-null  int64 
 14  HDL_Level                10000 non-null

In [7]:
df.describe()

,Patient_ID,Age,Diabetes,Hypertension,Obesity,Smoking,Alcohol_Consumption,Physical_Activity,Diet_Score,Cholesterol_Level,...,Diastolic_BP,Air_Pollution_Exposure,Family_History,Stress_Level,Healthcare_Access,Heart_Attack_History,Emergency_Response_Time,Annual_Income,Health_Insurance,Heart_Attack_Risk
count,10000.00000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,10000.000000,10000.000000
mean,5000.50000,49.394900,0.092900,0.24690,0.303700,0.301400,0.352800,0.595800,5.021700,224.753000,...,89.312000,0.403600,0.311300,5.518800,0.311000,0.152500,206.383400,1.022062e+06,0.344700,0.300700
std,2886.89568,17.280301,0.290307,0.43123,0.459878,0.458889,0.477865,0.490761,3.156394,43.359172,...,17.396486,0.490644,0.463048,2.866264,0.462926,0.359523,112.391711,5.605978e+05,0.475294,0.458585
min,1.00000,20.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,150.000000,...,60.000000,0.000000,0.000000,1.000000,0.000000,0.000000,10.000000,5.035300e+04,0.000000,0.000000
25%,2500.75000,35.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,2.000000,187.000000,...,74.000000,0.000000,0.000000,3.000000,0.000000,0.000000,110.000000,5.357838e+05,0.000000,0.000000
50%,5000.50000,49.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,5.000000,226.000000,...,89.000000,0.000000,0.000000,6.000000,0.000000,0.000000,206.000000,1.021383e+06,0.000000,0.000000
75%,7500.25000,64.000000,0.000000,0.00000,1.000000,1.000000,1.000000,1.000000,8.000000,262.000000,...,104.000000,1.000000,1.000000,8.000000,1.000000,0.000000,304.000000,1.501670e+06,1.000000,1.000000
max,10000.00000,79.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,10.000000,299.000000,...,119.000000,1.000000,1.000000,10.000000,1.000000,1.000000,399.000000,1.999714e+06,1.000000,1.000000


# Data Preprocessing

In [8]:
df.isnull().sum()

Patient_ID                 0
State_Name                 0
Age                        0
Gender                     0
Diabetes                   0
Hypertension               0
Obesity                    0
Smoking                    0
Alcohol_Consumption        0
Physical_Activity          0
Diet_Score                 0
Cholesterol_Level          0
Triglyceride_Level         0
LDL_Level                  0
HDL_Level                  0
Systolic_BP                0
Diastolic_BP               0
Air_Pollution_Exposure     0
Family_History             0
Stress_Level               0
Healthcare_Access          0
Heart_Attack_History       0
Emergency_Response_Time    0
Annual_Income              0
Health_Insurance           0
Heart_Attack_Risk          0
dtype: int64

In [9]:
len(df[df.duplicated()])

0

In [10]:
df = df.rename(columns= {
    'Patient_ID': 'id',
    'State_Name': 'state',
    'Alcohol_Consumption': 'alc',
    'Physical_Activity': 'phy',
    'Diet_Score': 'diet',
    'Cholesterol_Level': 'chol',
    'Triglyceride_Level': 'tgl',
    'LDL_Level': 'ldl',
    'HDL_Level': 'hdl',
    'Systolic_BP': 'sbp',
    'Diastolic_BP': 'dbp',
    'Air_Pollution_Exposure': 'air',
    'Family_History': 'fam',
    'Stress_Level': 'stress',
    'Healthcare_Access': 'hc_access',
    'Heart_Attack_History': 'ha_hist',
    'Emergency_Response_Time': 'ert',
    'Annual_Income': 'income',
    'Health_Insurance': 'insurance',
    'Heart_Attack_Risk': 'target',
    'Age': 'age',
    'Gender': 'gender',
    'Hypertension': 'hypertension',
    'Obesity': 'obesity',
    'Smoking': 'smoking',
    'Diabetes': 'diabetes'
})
df.head()

,id,state,age,gender,diabetes,hypertension,obesity,smoking,alc,phy,...,dbp,air,fam,stress,hc_access,ha_hist,ert,income,insurance,target
0,1,Rajasthan,42,Female,0,0,1,1,0,0,...,119,1,0,4,0,0,157,611025,0,0
1,2,Himachal Pradesh,26,Male,0,0,0,0,1,1,...,115,0,0,7,0,0,331,174527,0,0
2,3,Assam,78,Male,0,0,1,0,0,1,...,117,0,1,10,1,0,186,1760112,1,0
3,4,Odisha,58,Male,1,0,1,0,0,1,...,65,0,0,1,1,1,324,1398213,0,0
4,5,Karnataka,22,Male,0,0,0,0,0,1,...,109,0,0,9,0,0,209,97987,0,1


In [11]:
df['diet']

0       9
1       4
2       6
3       9
4       5
       ..
9995    6
9996    5
9997    2
9998    5
9999    8
Name: diet, Length: 10000, dtype: int64

In [12]:
df.columns

Index(['id', 'state', 'age', 'gender', 'diabetes', 'hypertension', 'obesity',
       'smoking', 'alc', 'phy', 'diet', 'chol', 'tgl', 'ldl', 'hdl', 'sbp',
       'dbp', 'air', 'fam', 'stress', 'hc_access', 'ha_hist', 'ert', 'income',
       'insurance', 'target'],
      dtype='object')

In [13]:
df.head()

,id,state,age,gender,diabetes,hypertension,obesity,smoking,alc,phy,...,dbp,air,fam,stress,hc_access,ha_hist,ert,income,insurance,target
0,1,Rajasthan,42,Female,0,0,1,1,0,0,...,119,1,0,4,0,0,157,611025,0,0
1,2,Himachal Pradesh,26,Male,0,0,0,0,1,1,...,115,0,0,7,0,0,331,174527,0,0
2,3,Assam,78,Male,0,0,1,0,0,1,...,117,0,1,10,1,0,186,1760112,1,0
3,4,Odisha,58,Male,1,0,1,0,0,1,...,65,0,0,1,1,1,324,1398213,0,0
4,5,Karnataka,22,Male,0,0,0,0,0,1,...,109,0,0,9,0,0,209,97987,0,1


# Data Visualization

In [14]:
# gender_counts = df['gender'].value_counts()
# plt.pie(gender_counts, labels=['Male', 'Female'], autopct='%.0f%%')
# plt.title("Distribution of patient's gender")
# plt.show()

In [15]:
# sn.countplot(data=df, y='state', order=df['state'].value_counts().index, palette="coolwarm")
# plt.title('Number of Patients in Each State')
# plt.xlabel('Count')
# plt.ylabel('State')
# plt.show()

In [16]:
# dff = df.copy()
# dff['age_group'] = pd.cut(dff['age'], bins=[20, 40, 60, 80], labels=["20-40", "40-60", "60-80"])

# age_counts = dff['age_group'].value_counts()
# plt.pie(age_counts, labels=["20-40", "40-60", "60-80"], autopct='%.0f%%')
# plt.title("Distribution of patient's age")
# plt.show()

In [17]:
df.head()

,id,state,age,gender,diabetes,hypertension,obesity,smoking,alc,phy,...,dbp,air,fam,stress,hc_access,ha_hist,ert,income,insurance,target
0,1,Rajasthan,42,Female,0,0,1,1,0,0,...,119,1,0,4,0,0,157,611025,0,0
1,2,Himachal Pradesh,26,Male,0,0,0,0,1,1,...,115,0,0,7,0,0,331,174527,0,0
2,3,Assam,78,Male,0,0,1,0,0,1,...,117,0,1,10,1,0,186,1760112,1,0
3,4,Odisha,58,Male,1,0,1,0,0,1,...,65,0,0,1,1,1,324,1398213,0,0
4,5,Karnataka,22,Male,0,0,0,0,0,1,...,109,0,0,9,0,0,209,97987,0,1


In [18]:
# sn.heatmap(df[['diabetes', 'obesity', 'phy', 'hypertension', 'chol', 'ldl', 'hdl', 'target']].corr(), annot=True, fmt=".2f")
# plt.title('Correlation Between Medical Risk Factors')
# plt.show()

In [19]:
# smoking_alcohol = df.groupby(['smoking', 'alc'])['id'].count().unstack()
# smoking_alcohol.plot(kind='bar', stacked=True)
# plt.title('Smoking & Alcohol Consumption Distribution')
# plt.xlabel('Smoking Status')
# plt.ylabel('Count')
# plt.legend(title='Alcohol Consumption')
# plt.show()

In [20]:
# plt.figure(figsize=(7, 5))
# sn.scatterplot(x='sbp', y='dbp', hue='target', data=df)
# plt.title('Systolic vs. Diastolic BP and Heart Attack Risk')
# plt.show()

In [21]:
# sn.barplot(x='target', y='air', data=df)
# plt.title('Air Pollution Exposure vs. Heart Attack Risk')
# plt.show()

In [22]:
# sn.barplot(x='target', y='fam', data=df)
# plt.title('Family History vs. Heart Attack Risk')
# plt.show()

In [23]:
# sn.barplot(x='target', y='stress', data=df)
# plt.title('Stress Level vs. Heart Attack Risk')
# plt.show()

In [24]:
# sn.barplot(x='target', y='hc_access', data=df)
# plt.title('Healthcare Access vs. Heart Attack Risk')
# plt.show()

In [25]:
# sn.countplot(data=df, x='hc_access', hue='target')
# plt.title('Healthcare Access vs. Heart Attack Risk')
# plt.show()

In [26]:
# sn.barplot(x='target', y='ha_hist', data=df)
# plt.title('Heart Attack History vs. Heart Attack Risk')
# plt.show()

In [27]:
# sn.barplot(x='target', y='ert', data=df)
# plt.title('Emergency Response Time vs. Heart Attack Risk')
# plt.show()

In [28]:
# sn.barplot(x='target', y='income', data=df)
# plt.title('Patient\'s Annual Income vs. Heart Attack Risk')
# plt.show()

In [29]:
# sn.barplot(x='target', y='insurance', data=df)
# plt.title('Patient\'s Health Insurance vs. Heart Attack Risk')
# plt.show()

In [30]:
# binary_vars = ['diabetes', 'hypertension', 'obesity', 'smoking', 'alc', 
#                'phy', 'fam', 'hc_access', 
#                'ha_hist', 'insurance']

# risk_rates = pd.DataFrame({
#     'Variable': binary_vars,
#     'Risk_Rate_0': [df[df[var] == 0]['target'].mean() for var in binary_vars],
#     'Risk_Rate_1': [df[df[var] == 1]['target'].mean() for var in binary_vars]
# })


# risk_diff = risk_rates['Risk_Rate_1'] - risk_rates['Risk_Rate_0']
# risk_diff_df = pd.DataFrame({'Variable': binary_vars, 'Risk_Difference': risk_diff})


# plt.figure(figsize=(7, 5))
# sn.barplot(x='Variable', y='Risk_Difference', data=risk_diff_df)
# plt.title('Difference in Heart Attack Risk for Binary Variables')
# plt.xticks(rotation=45)
# plt.show()

In [31]:
# numerical_vars = ['age', 'diet', 'chol', 'tgl', 
#                   'ldl', 'hdl', 'sbp', 'dbp', 
#                   'air', 'stress', 'ert', 
#                   'Aincome']

# corr_vars = [col for col in df.columns if col not in ['id', 'state', 'gender']]
# corr_matrix = df[corr_vars].corr()


# corr_stack = corr_matrix.stack().reset_index()
# corr_stack.columns = ['var1', 'var2', 'corr']
# high_corr_pairs = corr_stack[(abs(corr_stack['corr']) > 0.5) & (corr_stack['var1'] != corr_stack['var2'])]
# if high_corr_pairs.empty:
#     print("There are no highly correlated pairs with corr > 0.5 in the dataset.")
# else:
#     print("Highly correlated pairs found:")
#     print(high_corr_pairs)

In [32]:
# key_vars = ['age', 'chol', 'sbp', 'target']
# sn.pairplot(df[key_vars], hue='target')
# plt.show()

# Model

In [33]:
df.head()

,id,state,age,gender,diabetes,hypertension,obesity,smoking,alc,phy,...,dbp,air,fam,stress,hc_access,ha_hist,ert,income,insurance,target
0,1,Rajasthan,42,Female,0,0,1,1,0,0,...,119,1,0,4,0,0,157,611025,0,0
1,2,Himachal Pradesh,26,Male,0,0,0,0,1,1,...,115,0,0,7,0,0,331,174527,0,0
2,3,Assam,78,Male,0,0,1,0,0,1,...,117,0,1,10,1,0,186,1760112,1,0
3,4,Odisha,58,Male,1,0,1,0,0,1,...,65,0,0,1,1,1,324,1398213,0,0
4,5,Karnataka,22,Male,0,0,0,0,0,1,...,109,0,0,9,0,0,209,97987,0,1


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            10000 non-null  int64 
 1   state         10000 non-null  object
 2   age           10000 non-null  int64 
 3   gender        10000 non-null  object
 4   diabetes      10000 non-null  int64 
 5   hypertension  10000 non-null  int64 
 6   obesity       10000 non-null  int64 
 7   smoking       10000 non-null  int64 
 8   alc           10000 non-null  int64 
 9   phy           10000 non-null  int64 
 10  diet          10000 non-null  int64 
 11  chol          10000 non-null  int64 
 12  tgl           10000 non-null  int64 
 13  ldl           10000 non-null  int64 
 14  hdl           10000 non-null  int64 
 15  sbp           10000 non-null  int64 
 16  dbp           10000 non-null  int64 
 17  air           10000 non-null  int64 
 18  fam           10000 non-null  int64 
 19  stres

In [35]:
X = df.drop(['id', 'state', 'gender', 'diet', 'air', 'hc_access', 'ert', 'income', 'insurance', 'target'], axis=1)
y = df['target']

In [36]:
X.shape, y.shape

((10000, 16), (10000,))

In [37]:
X.columns

Index(['age', 'diabetes', 'hypertension', 'obesity', 'smoking', 'alc', 'phy',
       'chol', 'tgl', 'ldl', 'hdl', 'sbp', 'dbp', 'fam', 'stress', 'ha_hist'],
      dtype='object')

In [38]:
y.head()

0    0
1    0
2    0
3    0
4    1
Name: target, dtype: int64

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
preprocessor = Pipeline(steps=[('scaler', StandardScaler())])

In [41]:
models = {
    'Logistic Regression': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'AdaBoost': AdaBoostClassifier(),
    'LDA': LinearDiscriminantAnalysis(),
    'SVM': SVC(probability=True)
}

In [42]:
param_dists = {
    'Logistic Regression': {
        'model__C': loguniform(1e-3, 1e3),
        'model__solver': ['liblinear', 'saga'],
        'model__penalty': ['l1', 'l2']
    },
    'KNN': {
        'model__n_neighbors': randint(1, 50),
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]
    },
    'Naive Bayes': {
        'model__var_smoothing': loguniform(1e-9, 1e-1)
    },
    'Random Forest': {
        'model__n_estimators': randint(100, 1000),
        'model__max_depth': [None] + list(range(5, 30)),
        'model__min_samples_split': randint(2, 20),
        'model__min_samples_leaf': randint(1, 20)
    },
    'XGBoost': {
        'model__n_estimators': randint(100, 1000),
        'model__learning_rate': loguniform(1e-3, 0.3),
        'model__max_depth': randint(3, 10),
        'model__subsample': uniform(0.6, 0.4)
    },
    'AdaBoost': {
        'model__n_estimators': randint(50, 200),
        'model__learning_rate': loguniform(0.01, 1)
    },
    'LDA': {
        # LDA doesn't have many tunable hyperparameters; solver and shrinkage are optional.
        'model__solver': ['svd', 'lsqr'],
        'model__shrinkage': uniform(0, 1)
    },
    'SVM': {
        'model__C': loguniform(1e-3, 1e3),
        'model__kernel': ['linear', 'rbf', 'poly'],
        'model__gamma': loguniform(1e-4, 10)
    }
}


In [43]:
df['target'].value_counts()

target
0    6993
1    3007
Name: count, dtype: int64

In [44]:
X.shape, y.shape

((10000, 16), (10000,))

In [45]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [46]:
accuracy_results = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),  # Scaling step for numerical data
        ('model', model)
    ])
    
    # Use HalvingRandomSearchCV for faster hyperparameter tuning
    search = HalvingRandomSearchCV(
        pipeline,
        param_distributions=param_dists[model_name],
        factor=3,
        cv=cv,
        scoring='accuracy',
        random_state=42,
        n_jobs=-1,
        verbose=2  # Monitor progress
    )
    
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred) * 100
    accuracy_results[model_name] = accuracy
    


n_iterations: 6
n_required_iterations: 6
n_possible_iterations: 6
min_resources_: 12
max_resources_: 8000
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 666
n_resources: 12
Fitting 3 folds for each of 666 candidates, totalling 1998 fits
----------
iter: 1
n_candidates: 222
n_resources: 36
Fitting 3 folds for each of 222 candidates, totalling 666 fits
----------
iter: 2
n_candidates: 74
n_resources: 108
Fitting 3 folds for each of 74 candidates, totalling 222 fits
----------
iter: 3
n_candidates: 25
n_resources: 324
Fitting 3 folds for each of 25 candidates, totalling 75 fits
----------
iter: 4
n_candidates: 9
n_resources: 972
Fitting 3 folds for each of 9 candidates, totalling 27 fits
----------
iter: 5
n_candidates: 3
n_resources: 2916
Fitting 3 folds for each of 3 candidates, totalling 9 fits
n_iterations: 6
n_required_iterations: 6
n_possible_iterations: 6
min_resources_: 12
max_resources_: 8000
aggressive_elimination: False
factor: 3
----------
iter: 0
n_c

In [47]:
print("Model Accuracy Results:")
for model_name, accuracy in accuracy_results.items():
    print(f"{model_name}: {accuracy:.2f}%")

Model Accuracy Results:
Logistic Regression: 70.55%
KNN: 69.00%
Naive Bayes: 70.55%
Random Forest: 70.55%
XGBoost: 70.55%
AdaBoost: 70.55%
LDA: 70.55%
SVM: 70.55%
